# معمل مزودات الاستدلال ومراقبة الأداء: OpenAI و Groq و LangSmith
### Interactive Hands-on Lab: Multi-Provider Inference & LangSmith Tracing (Lectures 70 - 72)

أهلاً بك يا بطل في هذا المعمل التفاعلي!
في المحاضرات 70 و 71 و 72، تعرفنا على كيفية جلب المفاتيح الأساسية لمشاريع الوكلاء الذكية:
1. **OpenAI API Key**: النماذج التجارية المدفوعة (GPT-4o) من موقع `platform.openai.com`.
2. **Groq API Key**: النماذج مفتوحة المصدر فائقة السرعة مجاناً (Llama 3, Qwen) بتقنية الـ LPU من `console.groq.com`.
3. **LangSmith API Key**: منصة التتبع ورصد التكلفة وسرعة الاستجابة من `smith.langchain.com`.

سنتعلم ونجرب عملياً في هذا الدفتر:
- فحص جاهزية المفاتيح الثلاثة في ملف `.env`.
- تشغيل اختبار سرعة عملي بين النماذج.
- بناء جراف ذكي يوزع الأسئلة تلقائياً: الأسئلة السريعة تذهب لـ Groq LPU والأسئلة المعقدة تذهب للمسار الأعمق.
- قسم خاص: جرب بنفسك!


In [1]:
import os
import dotenv

dotenv.load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")
langsmith_key = os.getenv("LANGCHAIN_API_KEY") or os.getenv("LANGSMITH_API_KEY")

print("فحص جاهزية المفاتيح في ملف .env:")
print("- مفتاح Groq:", "متوفر وجاهز" if groq_key else "غير موجود")
print("- مفتاح OpenAI:", "متوفر وجاهز" if openai_key else "غير موجود")
print("- مفتاح LangSmith:", "متوفر وجاهز للتتبع" if langsmith_key else "غير مفعل حالياً")


فحص جاهزية المفاتيح في ملف .env:
- مفتاح Groq: متوفر وجاهز
- مفتاح OpenAI: غير موجود
- مفتاح LangSmith: غير مفعل حالياً


### 1. ما سر سرعة Groq الخارقة؟ ولماذا نستخدمه؟
يعتمد Groq على معالج مبتكر اسمه **LPU (Language Processing Unit)** بدلاً من كروت الشاشة التقليدية GPU.
الـ LPU يضع الأوزان في ذاكرة فائقة السرعة (SRAM)، مما يجعله يولد مئات الكلمات في الثانية وبشكل مجاني للمطورين!
تعال نجرب استدعاء سريعاً لنموذج Groq ونحسب الوقت المستغرق:


In [2]:
import time
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq

if groq_key:
    try:
        llm_groq = ChatGroq(model="openai/gpt-oss-120b", temperature=0.2, max_tokens=120)
        t0 = time.perf_counter()
        response = llm_groq.invoke([HumanMessage(content="اكتب جملة واحدة ملهمة عن الذكاء الاصطناعي.")])
        duration = time.perf_counter() - t0
        
        print(f"زمن الاستجابة: {duration:.3f} ثانية فقط!")
        print("رد النموذج:", response.content)
    except Exception as error:
        print(f"تم التقاط تنبيه استهلاك الحصة أو الاتصال: {error}")
        print("رد محاكى سريع: الذكاء الاصطناعي هو أداة تمكين العقل البشري لتوسيع آفاق الابتكار.")
else:
    print("تنبيه: مفتاح Groq غير متوفر، سنعمل بنمط المحاكاة المحلي.")


زمن الاستجابة: 0.522 ثانية فقط!
رد النموذج: الذكاء الاصطناعي هو الجسر الذي يربط أحلامنا اليوم بواقعٍ أكثر إبداعًا وابتكارًا غدًا.


### 2. ما فائدة LangSmith في مراقبة الجراف؟
عندما تبني وكيلاً يدور في حلقات تكرارية (Loops)، خدمة **LangSmith** تسمح لك برؤية شجرة القرار كاملة في واجهة بصرية:
- كم استهلك الوكيل من الرموز (Tokens)؟
- كم تكلفة كل خطوة بالسنت؟
- أين تعطل الوكيل إذا حدث خطأ؟


In [3]:
tracing = os.getenv("LANGCHAIN_TRACING_V2", "false")
project = os.getenv("LANGCHAIN_PROJECT", "rag-mastery-langgraph")

print(f"حالة التتبع المباشر (Tracing): {tracing}")
print(f"اسم المشروع في لوحة تحكم LangSmith: {project}")


حالة التتبع المباشر (Tracing): false
اسم المشروع في لوحة تحكم LangSmith: rag-mastery-langgraph


### 3. بناء جراف يوزع المهام بذكاء (Smart Dispatcher Graph)
الآن سنبني جرافاً يحتوي على:
1. عقدة تصنيف ذكية: تحدد هل السؤال خفيف وبسيط أم سؤال معقد؟
2. مسار سريع عبر Groq LPU للردود الفورية وتوفير التكلفة.
3. مسار عميق للأسئلة التحليلية الصعبة.


In [4]:
from typing import TypedDict, List, Literal, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, AIMessage

class RouterState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    route: str

def smart_router(state: RouterState):
    query = state["messages"][-1].content.lower()
    # إذا كان السؤال قصيراً أو ترحيباً نوجهه للمسار السريع
    if len(query.split()) < 6 or "مرحبا" in query or "صباح" in query:
        chosen_route = "fast_path"
    else:
        chosen_route = "deep_path"
    print(f"[المصنف]: تم توجيه السؤال إلى المسار -> {chosen_route}")
    return {"route": chosen_route}

def fast_worker(state: RouterState):
    print("[المسار السريع - Groq LPU]: جاري توليد إجابة فورية...")
    reply = AIMessage(content="[رد فوري سريع]: أهلاً بك! طلبك بسيط وتمت معالجته بسرعة فائقة عبر Groq.")
    return {"messages": [reply]}

def deep_worker(state: RouterState):
    print("[المسار العميق]: جاري التحليل التخصصي المعمق...")
    reply = AIMessage(content="[تحليل معمق]: تم فحص السؤال بعناية وتقديم إجابة هندسية مفصلة عبر المسار المتقدم.")
    return {"messages": [reply]}

def decide_path(state: RouterState) -> Literal["fast_worker", "deep_worker"]:
    if state["route"] == "fast_path":
        return "fast_worker"
    return "deep_worker"

print("تم تعريف دوال العقد والراوتر بنجاح!")


تم تعريف دوال العقد والراوتر بنجاح!


In [5]:
builder = StateGraph(RouterState)

builder.add_node("router", smart_router)
builder.add_node("fast_worker", fast_worker)
builder.add_node("deep_worker", deep_worker)

builder.add_edge(START, "router")
builder.add_conditional_edges("router", decide_path, {
    "fast_worker": "fast_worker",
    "deep_worker": "deep_worker"
})
builder.add_edge("fast_worker", END)
builder.add_edge("deep_worker", END)

app = builder.compile()

# تجربة سؤال خفيف
print("--- تجربة سؤال خفيف ---")
fast_res = app.invoke({"messages": [HumanMessage(content="مرحبا كيف حالك؟")]})
print("الرد:", fast_res["messages"][-1].content)

print("\n--- تجربة سؤال تحليلي مركب ---")
deep_res = app.invoke({"messages": [HumanMessage(content="اشرح لي بالتفصيل المعمارية الهندسية لمصفوفات الـ LPU مقارنة بالـ GPU")]})
print("الرد:", deep_res["messages"][-1].content)


--- تجربة سؤال خفيف ---
[المصنف]: تم توجيه السؤال إلى المسار -> fast_path
[المسار السريع - Groq LPU]: جاري توليد إجابة فورية...
الرد: [رد فوري سريع]: أهلاً بك! طلبك بسيط وتمت معالجته بسرعة فائقة عبر Groq.

--- تجربة سؤال تحليلي مركب ---
[المصنف]: تم توجيه السؤال إلى المسار -> deep_path
[المسار العميق]: جاري التحليل التخصصي المعمق...
الرد: [تحليل معمق]: تم فحص السؤال بعناية وتقديم إجابة هندسية مفصلة عبر المسار المتقدم.


### 4. جرب بنفسك! (Hands-on Experiments)
1. **تحدي 1**: ادخل على موقع `console.groq.com`، وأنشئ مفتاح API جديد وجربه في الخلية رقم 2 لتشاهد السرعة بنفسك!
2. **تحدي 2**: ادخل على `smith.langchain.com` وأنشئ حساباً مجانياً ثم انسخ المفتاح إلى ملف `.env` وافتح صفحة Projects لتشاهد تسجيل كل استدعاء حي!
3. **تحدي 3**: عدل دالة `smart_router` لتفحص هل السؤال يحتوي كلمة "كود" أو "برمجة" وتوجهه لمسار خاص بالبرمجة.
